# 🔬 实战分析:TODO Agent 的 9 条消息 trace 解读

这份 notebook 解读 [1_todo.ipynb](./1_todo.ipynb) 中 `result["messages"]` 的执行轨迹(来自 [1_a_output.txt](./1_a_output.txt))。

**用户输入**:
> Give me a short summary of the Model Context Protocol (MCP).

**总共产生 9 条消息**,完整地展示了 Manus 风格的 **"写计划 → 干活 → 复述 → 更新状态 → 回答"** 循环。这是对上一节 [1_a_todo.ipynb](./1_a_todo.ipynb) 中 `read_todos` 设计意图的**实战验证**。

## 一、9 条消息总览

```
[1] Human:    "Give me a short summary of MCP"
[2] AI    ── tool_calls = [write_todos(...), web_search(...)]    ← 并行 2 个!
[3] Tool   ← "Updated todo list to [...]"
[4] Tool   ← <MCP 搜索结果>
[5] AI    ── tool_calls = [read_todos()]                          ← 复述
[6] Tool   ← "Current TODO List: 1. 🔄 Search ... (in_progress)"
[7] AI    ── tool_calls = [write_todos(... status='completed')]  ← 标记完成
[8] Tool   ← "Updated todo list to [... completed]"
[9] AI    ── content = "## MCP - Summary ..."   tool_calls=[]    ← END
```

总共 **4 次 LLM 调用**(消息 2/5/7/9)、**4 次工具执行**(消息 3+4 是一次并行执行,消息 6 / 消息 8 各一次)。

## 二、关键观察 1:第一轮的并行调用 + 直接用 `in_progress`

看消息 2 的 `tool_calls`:

```python
tool_calls = [
    {'name': 'write_todos',
     'args': {'todos': [{'content': 'Search for info about MCP ...',
                         'status': 'in_progress'}]}},  # ⚠️ 直接 in_progress,而不是 pending
    {'name': 'web_search',
     'args': {'query': 'Model Context Protocol MCP'}}
]
```

这里有**两个聪明的优化**:

### (1) `write_todos` 和 `web_search` 被打包并行
LLM 看出来:"我写计划" 和 "我开始搜" 这两件事**没有依赖关系**(搜索的 query 不依赖 todos 的内容),所以一次塞了 2 个 tool_call 到同一个 AIMessage 里。`ToolNode` 收到后会**并发执行**这两个调用 → 产生消息 3 和消息 4。

> 这正是上一节 `((15+23)×64+52)/6` 例子里看到的**并行调用机制**,但这次用得更聪明 —— 没有 no-op,两个调用都是真有意义的活儿。

### (2) 跳过了 `pending` 状态
TODO 的状态本可以是 `pending → in_progress → completed`,但这里 LLM 直接写成 `in_progress`。原因是它**当下就要开始执行这个任务**(同一轮就发了 `web_search`),没必要先标 `pending` 再改 `in_progress`。

这呼应了 `WRITE_TODOS_DESCRIPTION` 里的规则:
> Only one in_progress task at a time

## 三、关键观察 2:`read_todos` 是"机械式遵守协议"的体现

看消息 5:

```python
AIMessage(tool_calls=[{'name': 'read_todos', 'args': {}, ...}])
```

**注意 `args={}`** —— LLM 没有传任何参数!这正是上一节讲的 `InjectedState` / `InjectedToolCallId` 在起作用:
- LLM 看到的 `read_todos` schema 里**没有参数**(state 和 tool_call_id 被剥掉了)
- 框架在 `ToolNode` 执行时**自己注入** state 和 tool_call_id

### 但这次调用其实是"过度的"

TODO 列表里**只有 1 个任务**,而它的状态(`Search ... in_progress`)在消息 3 里 800 token 之前才刚刚出现过。从信息论角度,LLM 完全不需要 `read_todos` 来 "提醒自己"。

**那为什么还要调用?** 因为系统 prompt(`TODO_USAGE_INSTRUCTIONS`)里**强制规定**了:
```
2. After you accomplish a TODO, use the read_todos to read the TODOs
   in order to remind yourself of the plan.
```

LLM 在**机械地遵守协议**。这恰恰证明了这个工具的设计意图:
- 在**简单任务**里看起来是浪费(就像现在这样)
- 在**长任务**里(20+ 轮工具调用)它就是救命的 anti-rot 机制

训练 / prompt 工程的策略是:**让 agent 始终遵守协议**,而不是"在长任务才用",因为 LLM 自己判断不准什么是"长任务"。

## 四、关键观察 3:`tool_call_id` 配对在所有消息间一致

Anthropic 的 tool_call_id 格式是 `toolu_...`(OpenAI 是 `call_...`)。配对关系:

| AIMessage 的 tool_call.id | 对应的 ToolMessage.tool_call_id |
|---|---|
| `toolu_01Uxn3KCFBvRZkiUkp8RBFhU` (write_todos in 消息 2) | 消息 3 |
| `toolu_01Pv2kTHyZBd2o8oEAdfbw7r` (web_search in 消息 2) | 消息 4 |
| `toolu_012qTeRM1maHhqPqr9D1TS3m` (read_todos in 消息 5) | 消息 6 |
| `toolu_01KQWb8MbPx5BizQGzrGRZ5N` (write_todos in 消息 7) | 消息 8 |

**注意消息 3 和 4 的顺序**:写 todo 的结果在前、搜索结果在后,跟消息 2 里 `tool_calls` 列表的顺序一致(虽然是并发执行,但 LangGraph 会按调用顺序回填结果到 messages)。

## 五、关键观察 4:Token 成本 —— recitation 的代价

提取每次 LLM 调用的 `input_tokens`:

| 轮次 | 消息 | input_tokens | 增量 | 增加的内容 |
|---|---|---|---|---|
| 1 | [2] | **1321** | — | system_prompt + user + tool schemas |
| 2 | [5] | **1647** | +326 | + 消息 [2] AIMessage + 消息 [3][4] 两条 ToolMessage(搜索结果) |
| 3 | [7] | **1727** | +80  | + 消息 [5] AIMessage + 消息 [6] ToolMessage(read_todos 输出) |
| 4 | [9] | **1856** | +129 | + 消息 [7] AIMessage + 消息 [8] ToolMessage(写 completed) |

**总 input_tokens = 1321 + 1647 + 1727 + 1856 = 6551**

### 跟"朴素 ReAct"对比一下

如果不用 TODO 工具,只用一个 `web_search`,这道题至少能省掉:
- 消息 2 里的 `write_todos` 调用(节省一次 tool_call 在 messages 里的字节数)
- 整个消息 5、6、7、8(4 条消息)
- **少 1-2 次 LLM 调用**

朴素版只需要 2 次 LLM 调用(决定调 web_search + 看到结果后总结),input_tokens 估计在 **1000 + 1200 ≈ 2200** 左右。

**结论**:在这个 1-step 任务里,TODO 协议让 token 成本增加了 **~3 倍**,LLM 调用次数翻倍。

**为什么还值得?** 因为这个开销在长任务里是**亚线性**的 —— 任务再复杂,recitation 的成本基本不变,而朴素 ReAct 的失败率(跑题、忘记目标)会指数上升。这是 **小成本买大保险** 的设计。

## 六、关键观察 5:Anthropic 模型的 stop_reason

用的是 `claude-sonnet-4-6`,所以 `response_metadata.stop_reason` 是 Anthropic 风格:

| 消息 | stop_reason | 含义 |
|---|---|---|
| [2] | `tool_use` | 因为要调用工具而停 → graph 继续走 `tools` 节点 |
| [5] | `tool_use` | 同上 |
| [7] | `tool_use` | 同上 |
| [9] | **`end_turn`** | 模型决定结束这一轮对话 → graph 走向 END |

OpenAI 的对应字段是 `tool_calls` 和 `stop`,LangChain 在 `AIMessage` 层面把它们**统一抽象**了:
- 都用 `tool_calls=[...]` 字段(空 list 等同于"不调工具")
- LangGraph 的 ReAct 路由器只看 `tool_calls` 是否为空,**不直接依赖 `stop_reason`**

所以同一份 graph 代码,你换成 OpenAI 模型也能跑,不用改一行 —— 这是 `init_chat_model` + LangChain 抽象的价值。

## 七、回应上一节:`read_todos` 真的把 todo 拉回了 context 末尾

对照消息 6 的内容:

```
Current TODO List:
1. 🔄 Search for information about Model Context Protocol (MCP) and provide a short summary (in_progress)
```

这条 ToolMessage 出现在消息 6,**就在 LLM 即将生成消息 7 之前**。也就是说:LLM 在决定"下一步该做什么"时,看到的**最后一条消息**就是它自己的 TODO 计划。

这正是上一节强调的:
> `read_todos` 把权威 state 里的计划**重新塞回 messages 末尾**,刷新 LLM 的注意力。

然后 LLM 看到 `🔄 in_progress` 状态,知道"这个任务我已经做完了,该标 completed"→ 于是产生消息 7。整个 **read → reflect → update** 的循环在这里完整闭合。

## 八、最终答案(消息 9)

```python
AIMessage(
    content='## Model Context Protocol (MCP) - Summary\n\n...',
    tool_calls=[],           # ← 🔑 空,触发 END
    response_metadata={'stop_reason': 'end_turn', ...},
)
```

这就是上一节讲的退出条件:**`tool_calls=[]` + `stop_reason='end_turn'` → graph 不再循环,返回最终结果**。

返回的内容是一个完整的 Markdown 格式总结,基于消息 4 的 `web_search` 结果。注意 LLM 并没有简单复制搜索结果,而是**重新组织**了内容(加了 "Key points" bullet list 和 "Think of it as a universal connector" 的类比),这是它"做完任务"的产出。

## 🎯 总结:这份 trace 验证了什么

| 概念 | 在 trace 里的体现 |
|---|---|
| **并行 tool_calls** | 消息 2 一次发了 2 个调用(write_todos + web_search) |
| **InjectedState 的隐形参数** | 消息 5 的 `read_todos` `args={}`,state 在框架内被注入 |
| **Command 同时更新 state + messages** | 消息 3 / 8 既是 ToolMessage,又触发了 `state["todos"]` 更新 |
| **tool_call_id 配对** | 4 对 AIMessage→ToolMessage 全部 id 匹配 |
| **ReAct 循环退出条件** | 消息 9 的 `tool_calls=[]` + `stop_reason='end_turn'` |
| **Manus recitation 模式** | 消息 5 强制 `read_todos` → 消息 6 把计划推到 context 末尾 → 消息 7 反思并 mark completed |
| **Token 成本随轮次增长** | 1321 → 1647 → 1727 → 1856(每轮重放完整 messages) |

### 一句话

> 这份 9 条消息的 trace 是一个**最小但完整**的 deep agent 演示:它跑了一遍 "plan → act → recite → reflect → answer" 的完整协议,即使任务简单到"杀鸡用牛刀",也忠实展示了为什么这套机制能在复杂任务里 work。